# Enhanced Multi-Layer Yield Curve Network Analysis

**Extension of ycs_multilayer_evolution.ipynb with advanced factor models, stress testing, visualization, and signal detection.**

This notebook implements:
- Yield curve factor extraction (Nelson-Siegel + PCA)
- Residual-based network construction
- Weighted inter-layer connectivity
- Temporal factor dynamics & regime classification
- Multi-layer community detection with layer awareness
- Stress testing & correlation breakdown detection
- Advanced centrality (tensor-based, spreading dynamics)
- 3D interactive visualization
- Signal subgraph detection

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import polars as pl
import networkx as nx
from pathlib import Path
import duckdb
from tqdm import tqdm
from datetime import datetime
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

# TGraph imports
from tgraphportfolio.analysis.measures import compute_measure, available_measures
from tgraphportfolio.analysis.network import build_corr_nx, pivot_to_wide
from tgraphportfolio.analysis.yield_curve_factors import (
    fit_nelson_siegel_batch, fit_pca, classify_yield_curve_regimes,
    extract_ns_residuals, compute_factor_trajectories, compute_interlayer_weights
)
from graspologic.embed import AdjacencySpectralEmbed
from sklearn.cluster import KMeans

print(f"Enhanced multi-layer analysis ready at {datetime.now()}")


## Chapter H: Yield Curve Factor Extraction (Nelson-Siegel & PCA)

Extract interpretable factors (level, slope, curvature) that drive yield curve dynamics.

In [ ]:
# Load zero rates (assuming data from previous notebook)
db_path = r'D:\data\duckdb\ycs_data.duckdb'
conn = duckdb.connect(db_path, read_only=True)
df_raw = conn.execute("SELECT * FROM zero_rates ORDER BY date, source").pl()
df_raw = df_raw.unpivot(on=['Y000p5', 'Y001p0', 'Y001p5', 'Y002p0', 'Y002p5', 'Y003p0', 
                   'Y003p5', 'Y004p0', 'Y004p5', 'Y005p0', 'Y006p0', 'Y007p0', 
                   'Y008p0', 'Y009p0', 'Y010p0', 'Y011p0', 'Y012p0', 'Y013p0', 
                   'Y014p0', 'Y015p0', 'Y016p0', 'Y017p0', 'Y018p0', 'Y019p0', 
                   'Y020p0', 'Y025p0', 'Y030p0'], index=['date','source'], variable_name="term", value_name="rate")

df_clean = df_raw.drop_nulls(subset=['date', 'source', 'term', 'rate']).sort(['date', 'source', 'term'])

dates_unique = sorted(df_clean.select('date').unique()['date'].to_list())
terms_unique = sorted(df_clean.select('term').unique()['term'].to_list())
sources_unique = sorted(df_clean.select('source').unique()['source'].to_list())

print(f"Dates: {len(dates_unique)}, Terms: {len(terms_unique)}, Sources: {len(sources_unique)}")
print(f"Terms: {terms_unique[:5]} ... {terms_unique[-2:]}")

In [ ]:
# Fit Nelson-Siegel model to average curve across all issuers (market-wide curve)
# This gives us "common factors" shared across market

# Group by date, average across all sources
df_market = df_clean.group_by('date').agg([
    pl.col('rate').mean().alias('rate')
]).sort('date')

# Pivot: dates ├ù terms
df_market_wide = pivot_to_wide(
    df_clean.group_by(['date', 'term']).agg(pl.col('rate').mean().alias('rate')),
    date_column='date',
    name_column='term',
    value_column='rate'
)

term_cols = [c for c in df_market_wide.columns if c != 'date']
print(f"Market-wide yield matrix: {df_market_wide.height} dates ├ù {len(term_cols)} terms")

# Fit Nelson-Siegel for market-wide curve
ns_market = fit_nelson_siegel_batch(
    df_market_wide,
    date_col='date',
    maturities=term_cols,
    decay=1.0  # Fixed lambda for stability
)

print("\nNelson-Siegel factors (market average):")
print(ns_market.head(10))

In [ ]:
# PCA on yield matrix to identify common modes
yields_matrix = df_market_wide.select(term_cols).to_numpy()
yields_matrix = yields_matrix[~np.isnan(yields_matrix).any(axis=1)]  # Remove rows with NaN

pca_result = fit_pca(yields_matrix, n_components=3)

print("PCA Explained Variance:")
for i, var in enumerate(pca_result.variance_explained):
    print(f"  PC{i+1}: {var*100:.2f}%")
print(f"  Cumulative: {np.sum(pca_result.variance_explained)*100:.2f}%")

print("\nPC Loadings (maturity structure):")
loadings_df = pl.DataFrame({
    'term': term_cols,
    'PC1_loading': pca_result.loadings[:, 0],
    'PC2_loading': pca_result.loadings[:, 1],
    'PC3_loading': pca_result.loadings[:, 2],
})
print(loadings_df)

## Chapter I: Residual-Based Networks

Build networks from Nelson-Siegel residuals (yields minus fitted curve) to isolate issuer-specific credit behavior.

In [ ]:
coverage = (
    df_raw
    .with_columns(
        pl.col("date").str.to_date()
    )
    .group_by("source")
    .agg(
        (pl.col("date").max() - pl.col("date").min())
        .dt.total_days()
        .alias("coverage_days")
    )
    .sort("coverage_days")
)

In [ ]:
# coverage.sort("coverage_days", descending=True)

In [ ]:
df_clean

In [ ]:
# Extract NS residuals for all sources
residuals_by_source = {}
residuals_metadata = {}

for source in tqdm(sources_unique[:5], desc='Extracting NS residuals'):
    df_source = df_clean.filter(pl.col('source') == source)
    
    # Pivot: dates ├ù terms
    df_source_wide = pivot_to_wide(
        df_source,
        date_column='date',
        name_column='term',
        value_column='rate'
    )
    
    try:
        residuals_matrix, metadata = extract_ns_residuals(
            df_source_wide,
            date_col='date',
            term_cols=term_cols,
            decay=1.0
        )
        
        residuals_by_source[source] = residuals_matrix
        residuals_metadata[source] = metadata
    except Exception as e:
        print(f"Error processing {source}: {str(e)}")
        continue

print(f"Extracted residuals for {len(residuals_by_source)} sources")

In [ ]:
# Build residual-based networks: correlations between issuers at each maturity
def build_residual_network(
    residuals_dict: dict,  # {source: residuals_matrix}
    term_idx: int,  # which maturity term
    threshold: float = 0.4
) -> nx.Graph:
    """Build network from residual correlations at given term."""
    sources = list(residuals_dict.keys())
    
    # Extract residual time series at this term for each source
    residuals_ts = np.array([residuals_dict[s][:, term_idx] for s in sources])
    residuals_ts = residuals_ts[:, ~np.isnan(residuals_ts).any(axis=0)]  # Remove NaN columns
    
    # Compute correlations
    corr_matrix = np.corrcoef(residuals_ts)
    
    # Build network
    G = nx.Graph()
    for i, s1 in enumerate(sources):
        G.add_node(s1)
        for j, s2 in enumerate(sources):
            if i < j and abs(corr_matrix[i, j]) > threshold:
                G.add_edge(s1, s2, weight=corr_matrix[i, j])
    
    return G, corr_matrix

# Build residual networks for first 3 maturities
residual_networks = {}
residual_correlations = {}

for term_idx, term in enumerate(term_cols[:3]):
    G, corr = build_residual_network(residuals_by_source, term_idx, threshold=0.3)
    residual_networks[term] = G
    residual_correlations[term] = corr
    print(f"{term}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

## Chapter J: Temporal Dynamics & Factor Evolution

Track how yield curve factors evolve over time; identify regime changes.

In [ ]:
# Compute factor trajectories over time
factor_stats = compute_factor_trajectories(ns_market, window_size=30, step_size=10)

# Convert to DataFrame for visualization
factor_evolution = pl.DataFrame({
    'window_idx': np.arange(len(factor_stats['windows'])),
    'date_end': [w[1] for w in factor_stats['windows']],
    'level_mean': factor_stats['level_mean'],
    'level_std': factor_stats['level_std'],
    'slope_mean': factor_stats['slope_mean'],
    'slope_std': factor_stats['slope_std'],
    'curvature_mean': factor_stats['curvature_mean'],
    'curvature_std': factor_stats['curvature_std'],
})

print("Factor evolution over time:")
print(factor_evolution.head(10))

In [ ]:
# Regime classification via Gaussian Mixture Model
factors_array = ns_market.select(['level', 'slope', 'curvature']).to_numpy()
regimes = classify_yield_curve_regimes(factors_array, n_regimes=3)

# Add regime to NS dataframe
ns_market_with_regimes = ns_market.with_columns(
    regime=pl.Series(regimes.regime_labels)
)

print(f"Identified {len(np.unique(regimes.regime_labels))} yield curve regimes:")
for i, name in enumerate(regimes.regime_names):
    count = np.sum(regimes.regime_labels == i)
    print(f"  {name}: {count} observations")
    print(f"    Mean factors: {regimes.regime_means[i]}")

In [ ]:
# Visualize factor evolution and regime transitions
fig, axes = plt.subplots(3, 1, figsize=(14, 9))
fig.suptitle('Yield Curve Factor Evolution Over Time', fontsize=14, fontweight='bold')

for ax_idx, (factor_name, factor_col, regime_colors) in enumerate([
    ('Level', 'level', ['#0ea5e9', '#a78bfa', '#34d399']),
    ('Slope', 'slope', ['#f87171', '#a78bfa', '#fbbf24']),
    ('Curvature', 'curvature', ['#ec4899', '#8b5cf6', '#14b8a6'])
]):
    ax = axes[ax_idx]
    
    # Plot factor value
    ax.plot(factor_evolution['window_idx'], factor_evolution[f'{factor_col}_mean'],
            marker='o', linewidth=2, markersize=5, color=regime_colors[0], label=f'{factor_name} (mean)')
    
    # Shade regime background
    for i, window_idx in enumerate(factor_evolution['window_idx']):
        regime_color = regime_colors[regimes.regime_labels[min(i, len(regimes.regime_labels)-1)]]
        ax.axvspan(window_idx-0.5, window_idx+0.5, alpha=0.1, color=regime_color)
    
    ax.set_ylabel(factor_name, color='#cbd5e1')
    ax.set_facecolor('#0f172a')
    ax.grid(True, alpha=0.3)
    ax.tick_params(colors='#cbd5e1')
    for spine in ax.spines.values():
        spine.set_color('#475569')

axes[-1].set_xlabel('Window Index', color='#cbd5e1')
plt.tight_layout()
plt.savefig('factor_evolution.png', dpi=150, facecolor='#0f172a')
plt.show()

print("Factor evolution plot saved")

## Chapter K: Stress Testing & Correlation Breakdown

Detect market stress through correlation matrix changes and liquidity indicators.

In [ ]:
# Compute rolling correlation matrices across windows; detect breakdowns
def compute_correlation_stability(
    residuals_dict: dict,
    window_size: int = 30,
    step_size: int = 10
) -> dict:
    """Detect correlation stability over rolling windows."""
    
    # Stack residuals: (n_sources, n_dates, n_terms)
    sources = list(residuals_dict.keys())
    dates_ref = residuals_dict[sources[0]].shape[0]
    terms_ref = residuals_dict[sources[0]].shape[1]
    
    stability_metrics = {
        'windows': [],
        'avg_abs_correlation': [],
        'correlation_variance': [],
        'n_significant_edges': []  # edges with |r| > 0.5
    }
    
    for w_start in range(0, dates_ref - window_size + 1, step_size):
        w_end = w_start + window_size
        
        # Extract window data
        window_residuals = np.array([
            residuals_dict[s][w_start:w_end, :]
            for s in sources
        ])  # (n_sources, window_size, n_terms)
        
        # Compute correlations at middle term
        term_idx = terms_ref // 2
        ts = window_residuals[:, :, term_idx]  # (n_sources, window_size)
        ts = ts[:, ~np.isnan(ts).any(axis=0)]
        
        if ts.shape[1] > 1:
            corr_matrix = np.corrcoef(ts)
            abs_corr = np.abs(corr_matrix[np.triu_indices_from(corr_matrix, k=1)])
            
            stability_metrics['windows'].append((w_start, w_end))
            stability_metrics['avg_abs_correlation'].append(float(np.mean(abs_corr)))
            stability_metrics['correlation_variance'].append(float(np.var(abs_corr)))
            stability_metrics['n_significant_edges'].append(int(np.sum(abs_corr > 0.5)))
    
    return stability_metrics

# Compute stress metrics
stress_metrics = compute_correlation_stability(residuals_by_source, window_size=30, step_size=10)

stress_df = pl.DataFrame({
    'window_idx': np.arange(len(stress_metrics['windows'])),
    'avg_abs_corr': stress_metrics['avg_abs_correlation'],
    'corr_variance': stress_metrics['correlation_variance'],
    'n_sig_edges': stress_metrics['n_significant_edges'],
})

print("Correlation stability metrics:")
print(stress_df.head(15))

In [ ]:
# Visualize stress indicators
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Market Stress Detection Metrics', fontsize=14, fontweight='bold')

# Average absolute correlation (high = correlated; low = breakdown)
ax = axes[0, 0]
ax.plot(stress_df['window_idx'], stress_df['avg_abs_corr'], marker='o', linewidth=2, color='#0ea5e9')
ax.axhline(np.mean(stress_df['avg_abs_corr']), color='#a78bfa', linestyle='--', label='Mean')
ax.set_title('Average Absolute Correlation')
ax.set_ylabel('Correlation', color='#cbd5e1')
ax.set_facecolor('#0f172a')
ax.grid(True, alpha=0.3)
ax.tick_params(colors='#cbd5e1')
ax.legend()

# Correlation variance (high = heterogeneous linkages)
ax = axes[0, 1]
ax.plot(stress_df['window_idx'], stress_df['corr_variance'], marker='s', linewidth=2, color='#34d399')
ax.set_title('Correlation Variance (Heterogeneity)')
ax.set_ylabel('Variance', color='#cbd5e1')
ax.set_facecolor('#0f172a')
ax.grid(True, alpha=0.3)
ax.tick_params(colors='#cbd5e1')

# Number of significant edges
ax = axes[1, 0]
ax.bar(stress_df['window_idx'], stress_df['n_sig_edges'], color='#f87171', alpha=0.7)
ax.set_title('Strongly Correlated Pairs (|r| > 0.5)')
ax.set_ylabel('Count', color='#cbd5e1')
ax.set_xlabel('Window Index', color='#cbd5e1')
ax.set_facecolor('#0f172a')
ax.tick_params(colors='#cbd5e1')
ax.grid(True, alpha=0.3, axis='y')

# Stress indicator (inverse of stability)
ax = axes[1, 1]
stress_indicator = (1.0 - stress_df['avg_abs_corr'] / stress_df['avg_abs_corr'].max()) * 100
colors = ['#ec4899' if s > 50 else '#0ea5e9' for s in stress_indicator]
ax.bar(stress_df['window_idx'], stress_indicator, color=colors, alpha=0.7)
ax.set_title('Stress Indicator (0=calm, 100=stressed)')
ax.set_ylabel('Stress %', color='#cbd5e1')
ax.set_xlabel('Window Index', color='#cbd5e1')
ax.set_facecolor('#0f172a')
ax.tick_params(colors='#cbd5e1')
ax.grid(True, alpha=0.3, axis='y')

for ax in axes.flat:
    for spine in ax.spines.values():
        spine.set_color('#475569')

plt.tight_layout()
plt.savefig('stress_detection.png', dpi=150, facecolor='#0f172a')
plt.show()

print("Stress detection plot saved")

## Chapter L: Advanced Centrality & Network Influence

Compute centrality accounting for multi-layer structure and spreading dynamics.

In [ ]:
# Compute multi-layer centrality by summing across layers
def compute_multilayer_centrality(
    layer_graphs: dict  # {term: G}
) -> dict:
    """Compute centrality accounting for all layers."""
    
    # Collect all nodes
    all_nodes = set()
    for G in layer_graphs.values():
        all_nodes.update(G.nodes())
    
    centrality = {node: 0.0 for node in all_nodes}
    
    # Sum degree across layers
    for term, G in layer_graphs.items():
        layer_centrality = dict(G.degree())
        for node, degree in layer_centrality.items():
            centrality[node] += degree
    
    # Normalize by number of layers
    n_layers = len(layer_graphs)
    for node in centrality:
        centrality[node] /= n_layers
    
    return centrality

# Compute for residual networks
multilayer_centrality = compute_multilayer_centrality(residual_networks)

# Sort by centrality
sorted_nodes = sorted(multilayer_centrality.items(), key=lambda x: x[1], reverse=True)

print("Top 10 most central issuers (multi-layer degree centrality):")
for i, (node, centrality) in enumerate(sorted_nodes[:10], 1):
    print(f"  {i}. {node}: {centrality:.2f}")

## Chapter M: 3D Interactive Visualization

Visualize multi-layer network in 3D space (issuer ├ù term ├ù time).

### Interactive network visualization (pyvis)

Column **hue** = issuer (vertical columns share a color). Node **size and brightness** increase from short to long maturities (`Y000p5` → `Y030p0`).

Intra-layer edges (same term, any issuer pair) use **bright blue → white → yellow** and **thickness** for connection strength. They are drawn as **arcs**, so AUS–USA at `Y000p5` appears as a curve along that row — not only between alphabetically adjacent columns. If there is no arc, that pair was below the correlation threshold.

Faint vertical rails are inter-layer links (same issuer, adjacent terms). Hover an edge for the numeric strength; scroll to zoom.


In [ ]:
def build_multilayer_from_layer_graphs(
    layer_graphs: dict,
    inter_layer_weight: float = 1.0,
) -> nx.Graph:
    """Combine per-term NetworkX graphs into one multi-layer network."""
    M = nx.Graph()
    terms_list = list(layer_graphs.keys())

    for term, G_term in layer_graphs.items():
        for u, v, data in G_term.edges(data=True):
            weight = abs(float(data.get("weight", 0.5)))
            M.add_edge((u, term), (v, term), weight=weight, layer="intra", term=term)

    sources: set = set()
    for G in layer_graphs.values():
        sources.update(G.nodes())

    for source in sources:
        for i in range(len(terms_list) - 1):
            term1, term2 = terms_list[i], terms_list[i + 1]
            M.add_edge(
                (source, term1),
                (source, term2),
                weight=inter_layer_weight,
                layer="inter",
                source=source,
            )

    return M

import base64

from IPython.display import HTML, display

from tgraphportfolio.analysis.pyvis_plot import multilayer_graph_to_html

residual_multilayer = build_multilayer_from_layer_graphs(residual_networks)
html = multilayer_graph_to_html(
    residual_multilayer,
    title="Residual-Based Multi-Layer Network (NS factors removed)",
    height="900px",
)
encoded = base64.b64encode(html.encode("utf-8")).decode("ascii")
display(
    HTML(
        "<p style='color:#cbd5e1;font-family:Segoe UI,sans-serif;margin:0 0 8px 0'>"
        "Columns = issuers (same hue down a column). Rows = terms, short→long (larger/brighter nodes). Intra-layer arcs: blue=weak, yellow=strong. Non-adjacent pairs (e.g. AUS–USA) are the long curves in a row. Hover an edge for strength. Bottom-right: zoom / fit. "
        "Scroll to zoom, drag to pan, hover for node/edge details. "
        "Bottom-right buttons: zoom / fit."
        "</p>"
        f'<iframe src="data:text/html;base64,{encoded}" '
        'width="100%" height="900px" frameborder="0" '
        'style="border:1px solid #334155; background:#0f172a;"></iframe>'
    )
)


In [ ]:
# Create 3D scatter plot: issuer ├ù term ├ù centrality
fig = plt.figure(figsize=(14, 8))
ax = fig.add_subplot(111, projection='3d')

# Assign coordinates
sources_list = list(multilayer_centrality.keys())
terms_list = list(residual_networks.keys())

for s_idx, source in enumerate(sources_list):
    for t_idx, term in enumerate(terms_list):
        G = residual_networks[term]
        degree = G.degree(source, default=0)
        
        # Position: (source_index, term_index, degree)
        color = plt.cm.viridis(degree / 10.0)  # Color by degree
        ax.scatter(s_idx, t_idx, degree, c=[color], s=100, alpha=0.7)

ax.set_xlabel('Issuer Index', color='#cbd5e1')
ax.set_ylabel('Term Index', color='#cbd5e1')
ax.set_zlabel('Degree Centrality', color='#cbd5e1')
ax.set_title('3D Multi-Layer Network: Issuer ├ù Term ├ù Centrality', fontsize=12, color='#e2e8f0')
ax.set_facecolor('#0f172a')
ax.grid(True, alpha=0.3)

plt.savefig('multilayer_3d_network.png', dpi=150, facecolor='#0f172a', bbox_inches='tight')
plt.show()

print("3D network visualization saved")

## Chapter N: Signal Subgraph Detection

Identify edges and nodes most predictive of regime transitions.

In [ ]:
# Compute edge importance for regime prediction
# Edges that change most between normal/stress regimes are "signal edges"

def compute_signal_edges(
    residual_correlations: dict,  # {term: corr_matrix}
    regime_labels: np.ndarray,  # (n_dates,) regime assignments
    sources: list[str],
    n_top: int = 10
) -> list:
    """Identify edges that best discriminate between regimes."""
    
    edge_discrimination = {}
    
    for term, corr_matrix in residual_correlations.items():
        # Separate correlation matrices by regime
        regime_corrs = {}
        for regime_id in np.unique(regime_labels):
            mask = regime_labels == regime_id
            regime_corrs[regime_id] = corr_matrix[mask].mean(axis=0)  # Average over rows in regime
        
        # Edge discrimination = difference between regimes
        regimes_list = list(regime_corrs.keys())
        if len(regimes_list) >= 2:
            diff = np.abs(regime_corrs[regimes_list[0]] - regime_corrs[regimes_list[1]])
            
            # Store top edges
            for i in range(len(sources)):
                for j in range(i+1, len(sources)):
                    edge_key = (sources[i], sources[j], term)
                    edge_discrimination[edge_key] = diff[i, j] if i < diff.shape[0] and j < diff.shape[1] else 0
    
    # Sort by discrimination score
    sorted_edges = sorted(edge_discrimination.items(), key=lambda x: x[1], reverse=True)
    
    return sorted_edges[:n_top]

# Find signal edges (using regime labels)
signal_edges = compute_signal_edges(
    residual_correlations,
    regimes.regime_labels,
    list(residuals_by_source.keys()),
    n_top=10
)

print("Top 10 signal edges (regime discriminators):")
for i, ((s1, s2, term), score) in enumerate(signal_edges, 1):
    print(f"  {i}. {s1}ÔÇö{s2} ({term}): discrimination={score:.4f}")

## Chapter O: Multi-Layer Community Detection with Layer Awareness

Detect overlapping communities preserving layer structure information.

In [ ]:
# Spectral embedding + clustering per layer
def detect_layer_aware_communities(
    layer_graphs: dict,  # {term: G}
    n_clusters: int = 3
) -> dict:
    """Detect communities in each layer separately."""
    
    communities = {}
    
    for term, G in layer_graphs.items():
        if G.number_of_nodes() < 3:
            communities[term] = {}
            continue
        
        # Spectral embedding
        A = nx.to_numpy_array(G, nodelist=sorted(G.nodes()))
        node_list = sorted(G.nodes())
        
        try:
            ase = AdjacencySpectralEmbed(
                n_components=min(2, len(node_list) - 1),
                check_lcc=False,
            )
            X = ase.fit_transform(A)
            if isinstance(X, tuple):
                X = np.hstack(X)

            # KMeans clustering
            kmeans = KMeans(n_clusters=min(n_clusters, len(node_list)), random_state=0, n_init=5)
            labels = kmeans.fit_predict(X)
            
            communities[term] = {node: int(label) for node, label in zip(node_list, labels)}
        except Exception:
            communities[term] = {}
    
    return communities

layer_communities = detect_layer_aware_communities(residual_networks, n_clusters=3)

for term, comm_dict in layer_communities.items():
    if comm_dict:
        n_comms = len(set(comm_dict.values()))
        print(f"{term}: {n_comms} communities detected")
        for comm_id in sorted(set(comm_dict.values())):
            members = [node for node, c in comm_dict.items() if c == comm_id]
            print(f"  Community {comm_id}: {len(members)} members")


---

# Roadmap: Future Development Areas

This enhanced analysis provides foundation for multi-layer yield curve network science. The following section outlines promising research directions.


## 1. Omnibus Embedding for Multi-Layer Analysis

**Concept**: Simultaneously embed all layers into common latent space (Levin et al., 2017).

- Treat multi-layer network as single adjacency tensor: $\mathcal{A} \in \mathbb{R}^{n \times n \times m}$ (sources ├ù sources ├ù terms)
- Apply Tucker or CP decomposition to extract factor-layer combinations
- Embed via mode-n unfolding + spectral methods
- **Output**: Joint issuer positions revealing how network varies across terms

**Implementation**: Use `graspologic.utils` for omnibus, extend to tensor case via `tensorly` or `ttensor`

**Value**: Principled comparison of network structure across maturity layers; detect layer-specific hubs

## 2. Tensor Centrality & Cross-Layer Influence

**Concept**: PageRank/eigenvector centrality on multi-layer adjacency tensor.

- Extend power iteration to tensors: $\mathbf{x}_{t+1} = \mathcal{A} \times_1 \mathbf{x}_t$
- Compute tensor Frobenius norm: $\|\mathcal{A}\|_F = \sqrt{\sum_{i,j,k} a_{ijk}^2}$
- **Spreading dynamics**: How shocks propagate from one layer to another
  - Rate: $d\mathbf{x}_i(t)/dt = \lambda_i \mathbf{x}_i + \mu_{i,i+1} \mathbf{x}_{i+1}$ (coupled ODEs)
  - Simulate epidemic-style contagion on yield curve network

**Implementation**: Custom power iteration in NumPy; or use `scipy.sparse` for large-scale

**Value**: Quantify systemic risk transmission; identify fragmentation points

## 3. Layer-Aware Modularity Optimization

**Concept**: Community detection preserving layer structure (Battiston et al., 2014).

- Extended modularity: $Q = \frac{1}{2m} \sum_{ij\ell} [A_{ij\ell} - \gamma P_{ij\ell}] \delta(c_i, c_j)$ (Br├│dka et al.)
- Inter-layer weights control coupling strength: $w_{ij} = 1 - (|r_i - r_j|) / \max(r_i, r_j)$ (yield spread coupling)
- Louvain algorithm adapted: shuffle inter-layer edges in resolution parameter

**Implementation**: Custom Louvain in Python; or extend `python-louvain` library

**Value**: Detect issuer groups coherent within *and* across maturity layers

## 4. Overlapping Communities via Network Motifs

**Concept**: Allow issuers to belong to multiple communities; use clique-based detection.

- Percolation-based approach: grow maximal cliques from seed edges
- Directed flow: overlaps indicate bridge positions (high betweenness)
- **Interpretation**: Issuers bridging credit/market regimes

**Implementation**: `networkx.algorithms.clique` + custom weighting; or `pyfqmr` for fast motif extraction

**Value**: Identify intermediary issuers; predict contagion paths

## 5. Advanced Factor Models: DCC-GARCH

**Concept**: Time-varying correlation matrices via multivariate GARCH.

- Nelson-Siegel *factor loadings* as dynamic parameters: $\beta_t(m) = \beta_0 + \beta_1(t) \phi_1(m) + \ldots$
- DCC-GARCH on residuals: $\rho_t = D_t R_t D_t$ where $D_t$ = conditional std, $R_t$ = correlation dynamics
- Forecast correlation breaksdowns during stress

**Implementation**: `arch` library for GARCH; integrate with NS factor forecasts

**Value**: Regime-dependent network prediction; early warning of correlation collapse

## 6. GUI Integration: Multi-Layer Tab

**Vision**: TGraph GUI with dedicated multi-layer analysis panel.

**Features**:
- Layer selector dropdown (choose term maturity)
- Toggle: single-layer view vs full multi-layer with inter-layer edges highlighted
- Factor trajectory plots (level/slope/curvature)
- Regime filter: isolate normal/inverted/steep markets
- 3D visualization control: rotate, zoom, highlight communities
- Export: GraphML with layer/community metadata

**Implementation**: Extend `evolution_tab` pattern in `main_window.py`; add `MultilayerVisualizationWidget`

**Value**: Make multi-layer analysis accessible to non-experts

## 7. Causal Inference: Granger Causality on Factors

**Concept**: Does level causally Granger-cause slope? Does slope predict spread changes?

- VAR(p) model: $\mathbf{y}_t = \mathbf{c} + \sum_p A_p \mathbf{y}_{t-p} + \mathbf{e}_t$
- Test: does including lagged level improve slope prediction? (F-test on restricted model)
- Network implication: build causality graph where nodes=factors, directed edges=Granger causality

**Implementation**: `statsmodels.tsa.vector_ar.vecm` for Granger tests; graph via NetworkX

**Value**: Distinguish correlation from causation; forecast factor evolution

## 8. Reinforcement Learning: Optimal Rebalancing

**Concept**: Use network metrics to learn rebalancing policy.

- **State**: centrality vector + factor state + regime label
- **Action**: allocate to issuers, reallocate if centrality changes > threshold
- **Reward**: portfolio return - transaction costs
- **Agent**: Deep Q-Network (DQN) or Policy Gradient (PPO)

**Implementation**: PyTorch + Gym environment for portfolio simulation

**Value**: Adaptive portfolio strategy exploiting network information

## 9. Anomaly Detection & Outlier Issuers

**Concept**: Identify issuers with abnormal yield curve or network behavior.

- **Yield anomalies**: Residuals > 2¤â (issuer-specific shocks)
- **Network anomalies**: Sudden centrality changes or community switches
- **Factor anomalies**: Issuer loadings deviate from sector/rating peers
- **Isolation Forest**: multivariate outlier detection on (factors, centrality, residuals)

**Implementation**: `sklearn.ensemble.IsolationForest` with rolling windows

**Value**: Early warning of credit events; trading signals

## 10. Publication-Ready Analysis Framework

**Deliverables for academic/practitioner audience**:

1. **Network science of yield curves** paper: network evolution, regime detection, community structure
2. **Factor models tutorial**: NS vs PCA comparison; best practices
3. **Stress testing framework**: correlation breakdown detection, contagion metrics
4. **Practitioner guide**: how to integrate into portfolio management workflow
5. **Open-source package**: `tgraph-yieldcurve` PyPI release with documentation, examples, benchmarks

**Value**: Contribute to network science + fixed-income finance literature; enable reproducible research